## MVP ENGENHARIA DE DADOS
Aluna: Bianca Goncalves de Oliveira

Matricula: 4052025002425

O presente trabalho tem como objetivo desenvolver um modelo de dados por meio da construção de pipelines de ETL utilizando a linguagem SQL. A arquitetura adotada está organizada nas camadas Bronze, Silver e Gold. A camada Bronze é responsável pela ingestão dos dados brutos; a camada Silver concentra os processos de tratamento, padronização e validação da qualidade dos dados; e a camada Gold disponibiliza dados consolidados e estruturados para análise e geração de insights.

A base utilizada neste estudo foi a disponibilizada publicamente pelo IMDb (Internet Movie Database), contendo informações sobre filmes, séries e documentários. A partir desses dados, foram realizados processos de tratamento e modelagem com o objetivo de criar, na camada Gold, um conjunto de informações capaz de responder a questões analíticas relacionadas aos gêneros cinematográficos e sua evolução ao longo dos anos.

As análises propostas buscam responder às seguintes perguntas:
- Qual é o ranking dos gêneros de filmes lançados em um determinado ano, considerando a média das avaliações recebidas?
- Quais são os 10 filmes com as maiores notas e quais seus respectivos anos de lançamento e gêneros?
- Dado um gênero específico, quais são os 50 filmes com as melhores avaliações e seus respectivos anos de lançamento?
- Quais são os anos que concentram o maior número de votos registrados, independentemente do gênero?


O trecho de código apresentado realiza a importação das bibliotecas necessárias para a obtenção, leitura e manipulação dos dados utilizados no projeto. A biblioteca Pandas, importada com o alias pd, é empregada para a análise e o processamento de dados estruturados em formato tabular. A biblioteca io fornece recursos para o gerenciamento de fluxos de dados em memória, facilitando a leitura e o tratamento das informações obtidas. Já a biblioteca requests é utilizada para realizar requisições HTTP, permitindo o acesso a dados hospedados em fontes externas, como os arquivos disponibilizados pelo IMDb (Internet Movie Database) por meio de URLs públicas.

Para este projeto, foram importadas duas tabelas da base de dados do IMDb. A primeira contém informações descritivas sobre os títulos, incluindo o título original e o título principal da obra, o tipo de conteúdo, a indicação de conteúdo adulto, o ano de lançamento, a duração da exibição e os gêneros associados. A segunda tabela reúne métricas de avaliação, contemplando a nota média atribuída a cada título e o respectivo número de votos recebidos pelos usuários.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Conforme mencionado anteriormente, o projeto foi estruturado em diferentes camadas, iniciando pela camada Bronze, responsável pela ingestão dos dados brutos. Como primeira etapa, realizou-se a exclusão da camada existente, com o objetivo de evitar conflitos decorrentes da presença de tabelas previamente criadas com os mesmos nomes. Em seguida, a camada foi recriada, garantindo um ambiente limpo e padronizado para o carregamento dos dados.

In [0]:
spark.sql("DROP DATABASE IF EXISTS bronze CASCADE")

In [0]:
spark.sql("CREATE DATABASE bronze")

In [0]:
title_basics_spark_df = (spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("compression", "gzip")
    .option("nullValue", "\\N")
    .csv("/Workspace/Users/oliveirabianca@id.uff.br/DbAcademy/title.basics.tsv.gz")
    .drop("isAdult"))

A coluna isAdult apresentou erro em sua criação e foi removida do DataFrame Spark, já que seus dados não seriam úteis para responder às perguntas da análise.

In [0]:

title_ratings_spark_df = (spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("compression", "gzip")
    .option("nullValue", "\\N")
    .csv("/Workspace/Users/oliveirabianca@id.uff.br/DbAcademy/title.ratings.tsv.gz"))

In [0]:
def run_with_retry(func, max_retries=1):
    for attempt in range(max_retries + 1):
        try:
            return func()
        except Exception as e:
            if "INACTIVITY_TIMEOUT" in str(e) and attempt < max_retries:
                global spark
                spark = DatabricksSession.builder.getOrCreate()
                continue
            raise

In [0]:
run_with_retry(lambda: title_basics_spark_df.write.mode("overwrite").saveAsTable("bronze.title_basics"))

In [0]:
run_with_retry(lambda: title_ratings_spark_df.write.mode("overwrite").saveAsTable("bronze.title_ratings"))

Por meio do comando SELECT DISTINCT, identificaram-se os tipos de títulos disponíveis na coluna selecionada, filtrando apenas os valores únicos. Como este trabalho delimita-se à análise de produções cinematográficas, apenas os tipos Movie, tvShort, short e tvMovie serão selecionados nas etapas posteriores.

In [0]:
%sql
SELECT * FROM bronze.title_basics LIMIT 20

In [0]:
%sql
SELECT DISTINCT titleType FROM bronze.title_basics;

In [0]:
%sql
SELECT * FROM bronze.title_ratings LIMIT 20

Após a conclusão das ações na camada Bronze (importação), a camada Silver é estruturada de forma semelhante para focar na análise e na verificação da consistência dos dados.

In [0]:
%sql
DROP DATABASE silver CASCADE

In [0]:
%sql
CREATE DATABASE silver

In [0]:
%sql
CREATE TABLE silver.title_ratings AS
SELECT * FROM bronze.title_ratings;

O primeiro passo consistiu na aplicação da operação de explode sobre a coluna genre, que originalmente continha múltiplos gêneros separados por vírgulas. Com essa transformação, cada gênero passou a ser representado em uma linha distinta, fazendo com que obras classificadas em mais de um gênero fossem replicadas em múltiplos registros. Como consequência, o identificador tconst, que anteriormente era único para cada título, deixou de apresentar unicidade na tabela resultante. Além disso, foram removidas colunas consideradas irrelevantes para os objetivos definidos no projeto, contribuindo para a simplificação e otimização do conjunto de dados utilizado nas análises

In [0]:
from pyspark.sql.functions import explode, split, col

title_genres_exploded_df = title_basics_spark_df.select(
    col("tconst"),
    col("primaryTitle"),
    col("titleType"),
    col("startYear"),
    explode(split(col("genres"), ",")).alias("genre")
)

title_genres_exploded_df.show(10)

In [0]:
title_genres_exploded_df.write.mode("overwrite").saveAsTable("silver.title_basics")

In [0]:
%sql
SELECT * FROM silver.title_basics LIMIT 100

Em seguida, executam-se consultas SQL nas tabelas para avaliar a qualidade dos dados.

In [0]:
%sql
SELECT COUNT(*) 
FROM silver.title_basics 
WHERE tconst IS NULL;

Não foram encontrados valores nulos em tconst.

In [0]:
%sql
SELECT *
FROM silver.title_basics
WHERE primaryTitle IS NULL OR TRIM(primaryTitle) = '' OR primaryTitle = '\N'

Foram identificadas algumas inconsistências na coluna primaryTitle e optou-se por alterar esses valores para "Unknown".

In [0]:
%sql
UPDATE silver.title_basics
SET primaryTitle = 'Unknown'
WHERE (primaryTitle IS NULL OR primaryTitle = '\N' OR TRIM(primaryTitle) = '')

In [0]:

%sql
SELECT COUNT(*) 
FROM silver.title_basics
WHERE titleType IS NULL OR TRIM(titleType) = '' OR primaryTitle = '\N';

Não foram encontrados valores nulos em titleType.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics 
WHERE startYear IS NULL OR startyear = '\N';

In [0]:
%sql
DELETE FROM silver.title_basics 
WHERE startYear IS NULL OR startyear = '\N';
     

Optou-se por deletar todos as linhas com valores de startYear nulos, já que o dado é essencial para a análise.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics 
WHERE startYear = '';
     

Os valores de startYear estavam armazenados como string, sendo necessaria a conversao para INT a fim de realizar as analises. 

In [0]:
%sql
ALTER TABLE silver.title_basics 
ADD COLUMN startYear_int INT;

UPDATE silver.title_basics 
SET startYear_int = CAST(startYear AS INT);
     

In [0]:
%sql
SELECT MIN(startYear_int), MAX(startYear_int) 
FROM silver.title_basics

Como o IMDB cataloga filmes que ainda náo foram lancados, optou-se por excluir qualquer filme com startYear de 2026 em diante. 

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE startYear_int > 2025

In [0]:
%sql
DELETE FROM silver.title_basics
WHERE startYear_int > 2025;

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE startYear_int > 2025

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE genre IS NULL OR genre = '/N' OR TRIM(genre) = '';

Como a variável de genero é essencial para as análises realizadas, os registros que nao continham essa informacao foram removidos da base de dados. 

In [0]:
%sql
SELECT genre, COUNT(*)
FROM silver.title_basics
GROUP BY genre
ORDER BY COUNT(*) DESC

Há um número expressivo de filmes e séries em cada um dos gêneros.

In [0]:
%sql
SHOW TABLES IN silver;

Concluídas as verificacoes na tabela de títulos, foram iniciadas as validacoes na tabela de avaliacoes. 

In [0]:
%sql
SELECT *
FROM silver.title_ratings LIMIT 50;

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE tconst IS NULL OR TRIM(tconst) = '' OR tconst = '\\N';

Nao há valores nulos em tconst.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE averageRating IS NULL OR TRIM(averageRating) = '';

Nao há valores nulos em avarageRating.

In [0]:
%sql
SELECT COUNT (*)
FROM silver.title_ratings
WHERE numVotes IS NULL OR TRIM(numVotes) = '';

Nao há valores nulos em numVotes.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE CAST(averageRating AS DOUBLE) < 0 OR CAST(averageRating AS DOUBLE) > 10;

Nao há notas fora do intervalo de 0 a 10.

In [0]:
%sql
SELECT count(*) 
FROM silver.title_ratings
WHERE numVotes = 0;

Nao há filmes com 0 numeros de votos.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings r
LEFT JOIN silver.title_basics b
ON r.tconst = b.tconst
WHERE b.tconst IS NULL;

Em seguida, foi verificda a correspondencia dos títulos entre as duas tabelas por meio da operacao JOIN.

In [0]:
%sql
CREATE TABLE silver.title_joined AS
SELECT
  b.tconst,
  b.titleType,
  b.primaryTitle,
  b.startYear,
  b.genre,
  r.averageRating,
  r.numVotes
FROM silver.title_ratings r
LEFT JOIN silver.title_basics b
ON r.tconst = b.tconst;

In [0]:
%sql
SELECT *
FROM silver.title_joined LIMIT 100;

Após a conclusao das atividades prevista para a camada prata, iniciou-se a criacao da camada ouro.

In [0]:
%sql
DROP DATABASE IF EXISTS gold CASCADE;

In [0]:
%sql
CREATE DATABASE gold;

O código a seguir realiza a criação da tabela final que será utilizada nas consultas analíticas. Inicialmente, são filtrados apenas os títulos dos tipos movie, tvShort, short e tvMovie, definidos como escopo da análise. Em seguida, a coluna startYear é renomeada para year, e são calculadas a média das avaliações e a soma da quantidade de votos para cada combinação de ano e gênero. Para isso, os registros são agrupados pelos atributos year e genre, consolidando aqueles que possuem os mesmos valores nessas colunas. Por fim, o resultado é ordenado por ano e gênero, facilitando sua análise e interpretação.

In [0]:
%sql
CREATE TABLE gold.genre_analysis AS
SELECT
    startYear AS year,
    genre,
    AVG(CAST(averageRating AS DOUBLE)) AS avgRating,
    SUM(CAST(numVotes AS INT)) AS totalVotes 
FROM
    silver.title_joined
WHERE
    titleType IN ('movie', 'tvShort', 'short', 'tvMovie')
GROUP BY
    startYear, 
    genre
ORDER BY
    startYear,
    genre;

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE year > 1980 LIMIT 500;

In [0]:
%sql
UPDATE gold.genre_analysis
SET avgRating = ROUND(avgRating, 3);

Os códigos apreentados a seguir realizam as consultas necessárias para responder as questoes inicialmente propostas.

In [0]:
%sql
SELECT DISTINCT *
FROM gold.genre_analysis
WHERE year = 1980 
ORDER BY avgRating DESC;

Neste caso, foi realizado um recorte para o ano de 1980 por meio da cláusula WHERE year = 1980, com o objetivo de identificar os gêneros que apresentaram as melhores avaliações médias. Em seguida, a cláusula ORDER BY avgRating DESC foi utilizada para ordenar os resultados da maior para a menor nota média. A análise dos dados evidencia que o gênero Documentary obteve a maior avaliação média, enquanto o gênero Horror apresentou o menor desempenho entre os gêneros considerados.

In [0]:
%sql
SELECT DISTINCT *
FROM gold.genre_analysis
WHERE year = 1990 
ORDER BY avgRating DESC;

Dez anos depois, em 1990, o gênero Documentary perdeu a primeira posição, sendo ultrapassado por Reality-TV. Entretanto, esse resultado deve ser analisado com cautela, uma vez que o número de votos associado a essa categoria é bastante reduzido. Além disso, considerando que a etapa de criação da tabela contemplou apenas títulos classificados como filmes, a presença do gênero Reality-TV pode indicar uma possível inconsistência na categorização dos dados. Observa-se ainda que a média de avaliações de Documentary apresentou crescimento em relação a 1980, enquanto a média do gênero Horror diminuiu, mantendo-o na última colocação entre os gêneros analisados.

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
ORDER BY avgRating DESC
LIMIT 10;

As maiores médias de avaliação são apresentadas nesta consulta. No entanto, observa-se um padrão de baixa quantidade de votos em várias das tuplas analisadas, o que reduz a representatividade dos resultados. Dessa forma, embora as notas médias sejam elevadas, elas devem ser interpretadas com cautela, pois foram calculadas a partir de uma amostra relativamente pequena, tornando essa métrica menos precisa e confiável.

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE totalVotes > 50000
ORDER BY avgRating DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE genre = 'Drama'
ORDER BY avgRating DESC
LIMIT 50;

Foi realizada uma análise específica do gênero Drama, com o objetivo de identificar os anos que apresentaram as maiores médias de avaliação. Observa-se que o ano de 1888 aparece entre os mais bem avaliados; entretanto, esse resultado deve ser interpretado com cautela, pois está associado a uma quantidade reduzida de votos. Dessa forma, a nota elevada pode ser influenciada pelo tamanho limitado da amostra, o que compromete sua representatividade em relação aos demais anos.

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE genre = 'Action'
ORDER BY avgRating DESC
LIMIT 50;

Em relação ao gênero Ação, observa-se uma tendência de avaliações mais elevadas entre os filmes lançados nos anos mais recentes.

In [0]:
%sql
SELECT year, SUM(totalVotes) AS total_votes_per_year
FROM gold.genre_analysis
GROUP BY year
ORDER BY total_votes_per_year DESC
LIMIT 50;

Por fim, o código apresentado identifica os anos que registraram o maior número de votos. Considerando a popularização da internet e a ampla disseminação de dispositivos eletrônicos, é natural que obras mais recentes tendam a acumular um volume maior de avaliações, em função do contexto tecnológico em que foram lançadas. Entretanto, o consumo de filmes não está restrito ao período de seu lançamento. Com a consolidação das plataformas de streaming, tornou-se mais fácil acessar e assistir a produções de diferentes épocas, permitindo que obras do século passado continuem sendo consumidas e avaliadas pelo público atualmente. Dessa forma, a quantidade de votos observada reflete não apenas a popularidade contemporânea dos títulos, mas também sua capacidade de permanecer relevante ao longo do tempo.

De maneira geral, avalio de forma bastante positiva a experiência de desenvolvimento deste trabalho. Antes de iniciar as etapas práticas, defini como objetivo aplicar os conceitos de análise e tratamento de dados aprendidos ao longo da disciplina, construindo um pipeline de dados capaz de transformar informações brutas em conhecimento útil para responder a questões analíticas. Considero que esse objetivo foi alcançado, pois consegui realizar desde a seleção da base de dados até a criação das consultas necessárias para extrair insights relevantes.

Inicialmente, minha intenção era utilizar uma base de dados relacionada a premiações do universo musical, como o Grammy Awards, para analisar características dos álbuns premiados e identificar possíveis padrões entre gêneros musicais. Entretanto, durante a busca por conjuntos de dados no Kaggle, encontrei dificuldades para localizar uma base consistente e adequada aos objetivos propostos. Além disso, surgiram desafios relacionados à qualidade e ao tratamento dos arquivos disponíveis. Diante desse cenário, optei por utilizar a base de dados do IMDb, que apresentava maior volume de informações, melhor estruturação e possibilitava a realização de análises semelhantes às planejadas inicialmente.

Durante a execução do projeto, alguns desafios demandaram decisões importantes sobre o tratamento dos dados. Um dos primeiros foi relacionado à coluna isAdult. Inicialmente, considerei corrigir as inconsistências existentes nesse campo; porém, como essa variável não fazia parte do escopo da análise, optei por removê-la. Embora essa decisão tenha simplificado o processo, reconheço que sua exclusão pode limitar futuras análises que eventualmente utilizassem essa informação. Outro ponto de atenção ocorreu quando a chave primária tconst perdeu sua unicidade após a etapa de expansão dos gêneros. Avaliei que essa situação era esperada, uma vez que um mesmo título pode estar associado a múltiplos gêneros, e que tal característica não comprometeria os objetivos definidos para o trabalho.

Também enfrentei algumas dificuldades na elaboração de consultas e transformações de dados ao longo do projeto. Em determinados momentos, foi necessário aprofundar meus estudos e consultar materiais de apoio para compreender melhor determinadas operações e implementar as soluções desejadas. Esse processo contribuiu significativamente para o meu aprendizado, pois permitiu consolidar conhecimentos práticos sobre manipulação e análise de dados.

Ao longo das análises, identifiquei limitações que podem ter influenciado alguns resultados obtidos. Uma delas está relacionada à quantidade de avaliações disponíveis para produções mais antigas. Em diversos casos, filmes lançados nos primeiros anos da base apresentavam um número reduzido de votos, o que resultava em médias elevadas, mas pouco representativas do conjunto geral de usuários. Dessa forma, para trabalhos futuros, seria interessante estabelecer critérios mínimos de quantidade de votos ou restringir o período analisado, aumentando a confiabilidade das conclusões obtidas.

Como possibilidade de evolução do projeto, também poderiam ser incorporadas novas variáveis presentes na base do IMDb, como informações sobre profissionais envolvidos nas produções, duração dos títulos ou aspectos relacionados à popularidade. Além disso, seria possível desenvolver visualizações interativas e aplicar técnicas estatísticas mais avançadas para aprofundar a análise dos padrões identificados.

Por fim, considero que a construção do MVP foi uma experiência extremamente enriquecedora. O projeto permitiu colocar em prática os conhecimentos adquiridos ao longo da disciplina e compreender, de forma concreta, todo o processo de transformação de dados brutos em informações relevantes para a tomada de decisão. Além do desenvolvimento técnico, a atividade proporcionou uma visão mais clara do potencial da análise de dados para gerar conhecimento e apoiar a solução de problemas reais, reforçando meu interesse pela área e ampliando minha compreensão sobre a importância de um processo estruturado de tratamento e análise de dados.